In [ ]:
import pandas as pd
from matplotlib_inline.backend_inline import set_matplotlib_formats
from matplotlib.pyplot import cm
from os import listdir as ls
from IPython.display import display, Markdown
import pycountry_convert as pc

from emu_renewal.inputs import OUTPUTS_PATH, get_standard_priors
from emu_renewal.outputs import get_param_vals_by_analysis, get_prop_improve
from emu_renewal.plotting import plot_kde_comparison, get_param_medians, AN_ABBREVS
from emu_renewal.utils import group_countries_by_continent, get_countries_by_continent

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import json
from emu_renewal.inputs import DATA_PATH

In [ ]:
plt.style.use("default")
set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "44255911"
countries = ls(job_path)

In [ ]:
def get_prop_improve(mob_type):
    param = "dispersion_proc"
    prop_improve = {}
    for c in countries:
        c_posts = get_param_vals_by_analysis(param, job_path / c)
        if mob_type in c_posts:
            prop_improve[c] = pd.Series(c_posts["no_mob"] > c_posts[mob_type]).astype(int).mean()
    return prop_improve

In [ ]:
prop_improve = get_prop_improve("g_mob")

In [ ]:
world = gpd.read_file(DATA_PATH / "mapping/ne_10m_admin_0_countries.shp")
world.loc[world["ADMIN"] == "France", "ISO_A3"] = "FRA"
world.loc[world["ADMIN"] == "Norway", "ISO_A3"] = "NOR"

In [ ]:
world["prop_improve"] = world["ISO_A3"].map(prop_improve)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 10))
world.boundary.plot(ax=ax, color="black", linewidth=0.2)
world.plot(column="prop_improve", ax=ax, cmap="Reds")